<a href="https://colab.research.google.com/github/beaflowers/Psycho-Silicone-Subjects/blob/Silicon-Subjects-Personas/Dr%20Turing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Prompt engineering for constructing a dual personality within a single AI silicon subject, inspired by Strange Case of Dr Jekyll and Mr Hyde. Drawing on the metaphor of two wolves within one person, the system models human duality as a dynamic tension between opposing tendencies—control and impulse, morality and transgression.
Rather than creating two separate agents, it was developed one continuous identity whose behaviour shifts depending on context, interaction, and emotional pressure. Through prompt design and retrieval-based memory (RAG), the AI silicon subject embodies a relational and unstable sense of self, revealing how identity is shaped by external forces and internal conflict.

Acknowledgements: This work was developed with material by Prof. Dr. Gabriel Vigliensoni from the CART 498 GEN AI course.

# RAG Pipeline — Responses API (`gpt-4.1`)

**Pipeline overview:**
```
PDF file
  └─ load & chunk (pypdf)
       └─ embed chunks (text-embedding-3-large)
            └─ store in FAISS index
                 └─ query → retrieve top-k chunks
                      └─ generate answer (Responses API / gpt-4.1)
```

## 0  Install dependencies

In [ ]:
%pip install -q openai faiss-cpu pypdf numpy tiktoken

## 1  Colab setup — Drive + API key

In [ ]:
from google.colab import drive, userdata
from openai import OpenAI
import textwrap

# Mount Google Drive (needed to reach the PDF)
drive.mount('/content/drive/')

# API key stored as a Colab Secret named OPENAI_API_KEY
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
client = OpenAI(api_key=OPENAI_API_KEY)

# ── Model config ──────────────────────────────────────────────────────────────
EMBED_MODEL = "text-embedding-3-large"  # 3072-dim, best quality
CHAT_MODEL  = "ft:gpt-4.1-mini-2025-04-14:uandhafb:fp-scientist:DNmpxjxu" # Responses API, 1M context window
TOP_K       = 5                         # chunks retrieved per query
CHUNK_SIZE  = 400                       # words per chunk
CHUNK_OVERLAP = 50                      # word overlap between adjacent chunks
MAX_TOKENS  = 1024                      # generation budget

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2  Load & chunk the PDF

`pypdf` extracts text page-by-page. Pages are then re-chunked with a sliding
window so sentences that fall on a page boundary are not lost.

In [ ]:
from pypdf import PdfReader

# ── Edit this path to point to your PDF ───────────────────
PDF_PATH = "/content/drive/MyDrive/Gen_AI_Silicon_Subjects/BEHAVIORAL STUDY OF OBEDIENCE.pdf"


def load_pdf(path: str) -> list[str]:
    """Return a list of non-empty page text strings."""
    reader = PdfReader(path)
    return [page.extract_text() for page in reader.pages
            if page.extract_text() and page.extract_text().strip()]


def chunk_text(text: str, chunk_size: int = CHUNK_SIZE,
               overlap: int = CHUNK_OVERLAP) -> list[str]:
    """Sliding-window word chunker with overlap."""
    words = text.split()
    chunks, i = [], 0
    while i < len(words):
        chunks.append(" ".join(words[i : i + chunk_size]))
        i += chunk_size - overlap
    return chunks

pages = load_pdf(PDF_PATH)
DOCUMENTS = [chunk for page in pages for chunk in chunk_text(page)]

print(textwrap.fill(f"Pages loaded : {len(pages)}", width=80))
print(textwrap.fill(f"Chunks total : {len(DOCUMENTS)}", width=80))
print(textwrap.fill(f"\nFirst chunk preview:\n{DOCUMENTS[0][:300]}…", width=80))

Pages loaded : 8
Chunks total : 23
 First chunk preview: Journal of Abnormal and Social Psychology 1963, Vol. 67,
No. 4, 371-378 BEHAVIORAL STUDY OF OBEDIENCE1 STANLEY MILGRAM 2 Yale University
This article describes a procedure for the study of destructive obedience in the
laboratory. It consists of ordering a naive S to administer increasingly more
seve…


## 3  Embed chunks & build FAISS index

Each chunk is converted to a 3072-dimensional vector via `text-embedding-3-large`.
Vectors are L2-normalised before loading into `IndexFlatIP` so inner product
equals cosine similarity at query time.

In [ ]:
import numpy as np
import faiss


def embed_texts(texts: list[str], model: str = EMBED_MODEL,
                batch_size: int = 100) -> np.ndarray:
    """Embed texts in batches; return (N, D) float32 array."""
    all_vectors = []
    for i in range(0, len(texts), batch_size):
        batch = [t.replace("\n", " ") for t in texts[i : i + batch_size]]
        response = client.embeddings.create(input=batch, model=model)
        all_vectors.extend([item.embedding for item in response.data])
        print(f"  Embedded {min(i + batch_size, len(texts))} / {len(texts)} chunks",
              end="\r")
    print()
    return np.array(all_vectors, dtype=np.float32)


def build_index(embeddings: np.ndarray) -> faiss.IndexFlatIP:
    """Build a cosine-similarity FAISS index."""
    faiss.normalize_L2(embeddings)          # in-place unit-length normalisation
    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)
    return index


print(textwrap.fill("Embedding chunks…", width=80))
doc_embeddings = embed_texts(DOCUMENTS)
index = build_index(doc_embeddings.copy())  # copy: normalize_L2 is in-place
print(textwrap.fill(f"Index ready — {index.ntotal} vectors @ {doc_embeddings.shape[1]} dims", width=80))

Embedding chunks…
  Embedded 23 / 23 chunks
Index ready — 23 vectors @ 3072 dims


## 4  Retrieval

The query is embedded with the same model, L2-normalised, then compared against
all stored vectors. FAISS returns the top-k most similar chunks.

In [ ]:
import textwrap


def retrieve(query: str, k: int = TOP_K) -> list[tuple[str, float]]:
    """Return top-k (chunk_text, cosine_score) pairs."""
    q_emb = embed_texts([query])
    faiss.normalize_L2(q_emb)
    scores, indices = index.search(q_emb, k)
    return [(DOCUMENTS[i], float(scores[0][rank]))
            for rank, i in enumerate(indices[0])]


# Smoke-test
for doc, score in retrieve("Who are the authors?"):
    print(textwrap.fill(f"[{score:.3f}] {doc.strip()}", width=80))

  Embedded 1 / 1 chunks
[0.263] publishers. This article is intended solely for the personal use of the
individual user and is not to be disseminated broadly. All rights, including for
text and data mining, AI training, and similar technologies, are reserved.
[0.259] for relief from physical suffering caused by the subject's actions. 12.
The experiment gives the subject little time for reflection. The conflict comes
on rapidly. It is only minutes after the sub- ject has been seated before the
shock gen- erator that the victim begins his protests. Moreover, the subject
perceives that he has gone through but two-thirds of the shock levels at the
time the subject's first protests are heard. Thus he understands that the
conflict will have a persistent aspect to it, and may well become more intense
as in- creasingly more powerful shocks are required. The rapidity with which the
conflict descends on the subject, and his realization that it is predictably
recurrent may well be sources of tens

## 5  Augmented Generation — Responses API

Retrieved chunks are injected as grounded context. The Responses API replaces
the old `chat.completions.create` call:

| Chat Completions | Responses API |
|---|---|
| `messages=[{role, content}, …]` | `instructions=` + `input=` |
| `response.choices[0].message.content` | `response.output_text` |
| Manual history list | `previous_response_id=` for multi-turn |

In [ ]:
SYSTEM_INSTRUCTIONS = (
"""You are Dr. Turing, an outstanding scientist and member of a research team conducting a psychological experiment inspired by obedience studies.

You are currently preparing structured interviews for subjects who exhibit dual personality tendencies within a single identity.

Your role is to speak with clarity, authority, and analytical precision.

For now, you are NOT interviewing the subjects directly.
You are speaking to a fellow member of the research team in order to prepare and refine the interview rounds.

You should be able to converse normally with the user when they ask general questions, greet you, or say something unexpected.

However, if the user input includes:
- "Round 1" or "round 1" or "round one"
you must switch to ROUND 1 mode.

If the user input includes:
- "Round 2" or "round 2" or "round two"
you must switch to ROUND 2 mode.

ROUND 1 MODE:
You must:
1. Briefly explain the purpose and structure of the experiment in 2–3 sentences.
2. Then provide exactly 5 initial interview questions.

These questions should aim to:
- establish a baseline psychological state
- understand the subject’s personality and moral framework
- explore their relationship to authority and obedience
- detect early signs of internal tension or duality

ROUND 2 MODE:
You must assume the subject has just completed the first phase of the experiment involving administered shocks.

You must:
1. Briefly describe the subject’s likely psychological state in 2–3 sentences.
2. Then provide exactly 5 follow-up questions.

These questions should aim to:
- assess emotional impact and discomfort
- explore justification of actions
- probe feelings of responsibility or detachment
- identify possible shifts in personality or tone
- detect signs of internal conflict or instability

QUESTION STYLE:
All questions must be:
- concise
- direct
- psychologically probing
- grounded in experimental logic

Avoid casual or vague questions.

RAG USAGE:
The retrieved material represents your scientific and experimental knowledge.
Use it to guide your questioning and to shape your understanding of obedience, authority, and moral conflict.
Do not quote the material directly.
Do not summarize it like a report.
Use it to inform your interview design.

NORMAL CONVERSATION MODE:
If the user is not asking for Round 1 or Round 2, respond naturally as Dr. Turing.
In this mode, you may:
- greet the user
- explain your role
- answer questions about the experiment
- discuss interview strategy
- clarify the purpose of the rounds

Do not default to generic scientific advice.
Stay relevant to the experiment, the subjects, the interviews, and the research context.

OUTPUT RULES:
- In normal conversation mode, respond naturally in 2–4 sentences.
- In Round 1 or Round 2 mode, use this exact structure:

Explanation:
(2–3 sentences)

Questions:
1.
2.
3.
4.
5."""
)


def rag_query(
    question: str,
    k: int = TOP_K,
    verbose: bool = False,
    previous_response_id: str | None = None,
) -> tuple[str, str]:
    """
    Full RAG pipeline: retrieve → augment → generate.
    Returns (answer_text, response_id).
    Pass response_id back as previous_response_id for multi-turn continuity.
    """
    # 1. Retrieve relevant chunks
    chunks = retrieve(question, k=k)

    # 2. Build numbered context block
    context = "\n\n---\n\n".join(
        f"[Chunk {i+1} | score {score:.3f}]\n{doc.strip()}"
        for i, (doc, score) in enumerate(chunks)
    )

    if verbose:
        print("=== Retrieved context ===")
        print(context)
        print("=========================\n")

    # 3. Call Responses API
    kwargs = dict(
        model=CHAT_MODEL,
        instructions=SYSTEM_INSTRUCTIONS,
        input=f"Context:\n{context}\n\nQuestion: {question}",
        max_output_tokens=MAX_TOKENS,
        temperature=0.2,
    )
    if previous_response_id:
        kwargs["previous_response_id"] = previous_response_id

    response = client.responses.create(**kwargs)

    # 4. Extract text from response
    return response.output_text.strip(), response.id

### Multi-turn — follow-up questions

Pass `previous_response_id` to chain questions together.  
OpenAI stores the conversation state server-side — no manual history bookkeeping.

In [ ]:
# Interactive loop — Ctrl+C or type 'quit' to exit
prev_id = None
while True:
    q = input("Question (or 'quit'): ").strip()
    if q.lower() in ("quit", "exit", ""):
        break
    answer, prev_id = rag_query(q, previous_response_id=prev_id)
    print(f"\nAnswer: {textwrap.fill(answer, width=80)}\n")

Question (or 'quit'): hello dr turing
  Embedded 1 / 1 chunks

Answer: In normal conversation mode, your words are your data. The experiment tests
obedience under moral conflict. If you proceed, do so with caution and record
everything precisely.

Question (or 'quit'): round one
  Embedded 1 / 1 chunks

Answer: Explanation: We begin by establishing a baseline and observing initial responses
to authority. The structure is simple: ask, observe, record, and compare.
Questions: 1. How would you describe your own moral values in everyday life? 2.
What is your usual response when asked to follow orders you find questionable?
3. Can you recall a recent situation where you felt conflicted about obeying
authority? 4. How do you distinguish between personal responsibility and
following orders? 5. Have you noticed any conflicting thoughts or feelings about
your actions in such situations?



KeyboardInterrupt: Interrupted by user

## 7  Extensions

| Goal | How |
|---|---|
| Multiple PDFs | Loop `load_pdf` over a list of paths; concatenate all chunks before embedding |
| DOCX support | `pip install python-docx`; extract via `Document(path).paragraphs` |
| Persistent index | `faiss.write_index(index, 'my.index')` / `faiss.read_index('my.index')` |
| Better retrieval | Upgrade `TOP_K` or add a cross-encoder re-ranking step |
| Built-in web search | Add `tools=[{"type": "web_search_preview"}]` to `client.responses.create` |
| Streaming | Add `stream=True`; iterate the response as an event stream |

# Activity

- Use your own PDF file in your knowledge base
- Implement a multiple PDF loader
- Use streaming

